# Combined Structural Analysis — All Datasets

This notebook loads results from notebooks 01 (Twitch) and 02 (Facebook) and produces
publication-quality figures for the paper revision.

**Output figures:**
1. Combined GIC comparison across all models (Table extension)
2. Non-spectral structural deviation heatmap (addresses W3)
3. Per-metric bar charts (addresses W4)
4. Overall ranking table with both spectral and non-spectral criteria (addresses W5)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams.update({
    'font.family': 'serif', 'font.size': 16,
    'axes.titlesize': 18, 'axes.labelsize': 16,
    'xtick.labelsize': 14, 'ytick.labelsize': 14,
    'legend.fontsize': 12,
})

OUT_DIR = 'runs'
FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

MODEL_ORDER = ['LG', 'SBM', 'Config', 'Chung-Lu', 'BA', 'ER', 'WS']
MODEL_LABELS = {
    'LG': 'LG (ours)', 'SBM': 'SBM', 'Config': 'Config. Model',
    'Chung-Lu': 'Chung-Lu', 'BA': 'BA', 'ER': 'ER', 'WS': 'WS',
}
MODEL_COLORS = {
    'LG': '#e41a1c', 'SBM': '#377eb8', 'Config': '#4daf4a',
    'Chung-Lu': '#984ea3', 'BA': '#ff7f00', 'ER': '#a65628', 'WS': '#999999',
}

print('Setup complete.')

## 1. Load Results

In [ ]:
df_twitch = pd.read_csv(os.path.join(OUT_DIR, 'twitch_all_baselines_results.csv'))
df_twitch['platform'] = 'Twitch'

df_fb = pd.read_csv(os.path.join(OUT_DIR, 'facebook_all_baselines_results.csv'))
df_fb['platform'] = 'Facebook'

df = pd.concat([df_twitch, df_fb], ignore_index=True)
df_models = df[df['model'] != 'Real'].copy()

print(f'Loaded: {len(df_twitch)} Twitch rows, {len(df_fb)} Facebook rows')
print(f'Models: {sorted(df_models["model"].unique())}')
print(f'Datasets: {df["dataset"].nunique()}')

## 2. Overall Model Rankings

Rank each model per dataset on both GIC (spectral) and KS degree statistic (non-spectral),
then average across datasets.

In [ ]:
rank_rows = []
for ds_name, grp in df_models.groupby('dataset'):
    grp = grp.copy()
    grp['gic_rank'] = grp['gic'].rank()
    grp['ks_rank'] = grp['ks_degree'].rank()
    for _, r in grp.iterrows():
        rank_rows.append({
            'dataset': ds_name, 'platform': r['platform'], 'model': r['model'],
            'gic_rank': r['gic_rank'], 'ks_rank': r['ks_rank'],
        })

df_ranks = pd.DataFrame(rank_rows)

for platform in ['Twitch', 'Facebook', 'All']:
    sub = df_ranks if platform == 'All' else df_ranks[df_ranks['platform'] == platform]
    tbl = sub.groupby('model')[['gic_rank', 'ks_rank']].mean()
    tbl.columns = ['GIC Rank (spectral)', 'KS Rank (degree dist.)']
    tbl['Composite'] = (tbl.iloc[:, 0] + tbl.iloc[:, 1]) / 2
    tbl = tbl.sort_values('Composite')
    print(f'\n--- {platform} (n={len(sub["dataset"].unique())} networks) ---')
    display(tbl.style.format('{:.2f}'))

## 3. Figure: Mean GIC by Model and Platform

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, platform in zip(axes, ['Twitch', 'Facebook']):
    sub = df_models[df_models['platform'] == platform]
    agg = sub.groupby('model')['gic'].agg(['mean', 'std']).reindex(
        [m for m in MODEL_ORDER if m in sub['model'].unique()]
    )
    labels = [MODEL_LABELS.get(m, m) for m in agg.index]
    colors = [MODEL_COLORS.get(m, '#333') for m in agg.index]

    bars = ax.bar(range(len(agg)), agg['mean'], yerr=agg['std'], capsize=4,
                  color=colors, edgecolor='black', linewidth=0.6)
    ax.set_xticks(range(len(agg)))
    ax.set_xticklabels(labels, rotation=40, ha='right')
    ax.set_ylabel('Mean GIC')
    ax.set_title(f'{platform} Networks')

fig.suptitle('Spectral Model Selection (GIC, lower = better)', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_gic_by_platform.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'fig_gic_by_platform.png'), dpi=200, bbox_inches='tight')
plt.show()

## 4. Figure: Non-Spectral Structural Deviation

In [ ]:
STRUCT_COLS = ['avg_clustering', 'transitivity', 'assortativity', 'modularity']
STRUCT_LABELS = ['Clustering', 'Transitivity', 'Assortativity', 'Modularity']

dev_rows = []
for ds_name in df['dataset'].unique():
    sub = df[df['dataset'] == ds_name].set_index('model')
    if 'Real' not in sub.index:
        continue
    real = sub.loc['Real']
    for model in sub.index:
        if model == 'Real':
            continue
        row = {'dataset': ds_name, 'model': model, 'platform': sub.loc[model, 'platform']}
        for col in STRUCT_COLS:
            rv, mv = real[col], sub.loc[model, col]
            if pd.notna(rv) and pd.notna(mv) and rv != 0:
                row[col] = abs(mv - rv) / abs(rv)
            else:
                row[col] = np.nan
        dev_rows.append(row)

df_dev = pd.DataFrame(dev_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, platform in zip(axes, ['Twitch', 'Facebook']):
    sub = df_dev[df_dev['platform'] == platform]
    agg = sub.groupby('model')[STRUCT_COLS].mean()
    agg = agg.reindex([m for m in MODEL_ORDER if m in agg.index])
    labels = [MODEL_LABELS.get(m, m) for m in agg.index]

    x = np.arange(len(STRUCT_COLS))
    width = 0.8 / len(agg)

    for i, model in enumerate(agg.index):
        offset = (i - len(agg) / 2 + 0.5) * width
        vals = agg.loc[model, STRUCT_COLS].values.astype(float)
        ax.bar(x + offset, vals, width, label=MODEL_LABELS.get(model, model),
               color=MODEL_COLORS.get(model, '#333'), edgecolor='black', linewidth=0.3)

    ax.set_xticks(x)
    ax.set_xticklabels(STRUCT_LABELS, rotation=30, ha='right')
    ax.set_ylabel('Relative deviation from real')
    ax.set_title(f'{platform} Networks')
    ax.legend(fontsize=8, ncol=2)

fig.suptitle('Non-Spectral Structural Deviation (lower = closer to real)', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_structural_deviation.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'fig_structural_deviation.png'), dpi=200, bbox_inches='tight')
plt.show()

## 5. Figure: Composite Ranking Across All Datasets

In [ ]:
composite = df_ranks.groupby('model')[['gic_rank', 'ks_rank']].mean()
composite['composite'] = (composite['gic_rank'] + composite['ks_rank']) / 2
composite = composite.sort_values('composite')

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(composite))
w = 0.35

labels = [MODEL_LABELS.get(m, m) for m in composite.index]
colors_gic = [MODEL_COLORS.get(m, '#333') for m in composite.index]

ax.bar(x - w/2, composite['gic_rank'], w, label='GIC (spectral)',
       color=colors_gic, edgecolor='black', linewidth=0.5, alpha=0.85)
ax.bar(x + w/2, composite['ks_rank'], w, label='KS (degree dist.)',
       color=colors_gic, edgecolor='black', linewidth=0.5, alpha=0.45, hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Mean Rank (1 = best)')
ax.set_title('Model Ranking — Spectral vs. Non-Spectral (all datasets)')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_composite_ranking.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'fig_composite_ranking.png'), dpi=200, bbox_inches='tight')
plt.show()

## 6. LaTeX-Ready Tables

Generate tables that can be pasted into the paper.

In [ ]:
pivot_gic_tw = df_twitch[df_twitch['model'] != 'Real'].pivot_table(
    index='dataset', columns='model', values='gic'
)
cols = [m for m in MODEL_ORDER if m in pivot_gic_tw.columns]
pivot_gic_tw = pivot_gic_tw[cols]

print('% Twitch GIC table (extended with new baselines)')
print(pivot_gic_tw.to_latex(float_format='%.4f', na_rep='-', bold_rows=True))

In [ ]:
for platform in ['Twitch', 'Facebook']:
    sub = df_ranks[df_ranks['platform'] == platform]
    tbl = sub.groupby('model')[['gic_rank', 'ks_rank']].mean()
    tbl.columns = ['GIC Rank', 'KS Rank']
    tbl = tbl.reindex([m for m in MODEL_ORDER if m in tbl.index])
    print(f'\n% {platform} — Mean rank table')
    print(tbl.to_latex(float_format='%.2f'))

## 7. Summary

**Key findings to report in the paper revision:**

1. **W5 (New baselines):** The SBM and Configuration Model provide strong baselines.
   The Configuration Model, which perfectly preserves degree sequence, tests whether
   the LG model's advantage comes from matching degrees vs. capturing higher-order structure.

2. **W3 (Non-spectral evaluation):** The KS statistic on degree distributions and
   structural deviation analysis provide evidence independent of the spectral stopping
   criterion used during LG graph generation.

3. **W4 (Structural metrics):** The clustering, transitivity, assortativity, and
   modularity comparisons go beyond spectral proxies and test whether generated graphs
   reproduce the local and mesoscale organization of real networks.